# rerank_llm_feature — Qwen judge yes/no scores on R (the judge finally sees eligibility)

Scores each (topic, doc) with Qwen2.5-7B as `logsumexp(yes) − logsumexp(no)` on a single forward pass,
using the **frozen `elig_first-L512`** representation via `llm_yesno_scores` → so the judge sees the
eligibility criteria — unlike §11c, whose null was measured on the eligibility-blind 1800-char text.
Writes `llm_scores_R.jsonl` (the ensemble `llm_yesno` feature) and reports standalone judged-pool NDCG
under R, re-opening the §11c question: is the judge better when it can actually read eligibility?


## Setup (Colab — GPU, ~14GB for Qwen-7B fp16)


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers accelerate datasets pandas tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
DATA_ROOT = '/content/drive/MyDrive/ct_data23'
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval, llm_yesno_scores, ndcg_at_k
POOL_TAG = 'R'   # set 'nqs' to re-score on the NQS pool (needs nqs_retrieval first)
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag=POOL_TAG)   # frozen R
print('repr:', cfg.repr_tag(), '| judge:', cfg.llm_ckpt)


In [ ]:
# Candidate pool. 'retrieval' = the eval_fullcorpus pool (top llm_top_k) -> the ENSEMBLE feature.
# 'judged' = the judged pool -> the standalone §11c check you already ran (0.65 on trec22).
POOL_SOURCE = 'retrieval'   # 'retrieval' (needs cfg.pool_path()) | 'judged'
SETS = ['trec21', 'kz', 'trec22']
OUT_PATH = cfg.feat_file('llm_scores')
corpus_ids, corpus_fields = load_corpus(cfg)
id2fields = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, SETS)
if POOL_SOURCE == 'retrieval':
    rp = json.load(open(cfg.pool_path()))
    pools = {s: {t: [d for d in docs[:cfg.llm_top_k] if d in id2fields] for t, docs in rp[s].items()} for s in SETS}
else:
    pools = {s: {t: [d for d in sets[s]['rel_dict'][t] if d in id2fields]
                 for t in sets[s]['rel_dict'] if t in sets[s]['topic2text']} for s in SETS}
print(POOL_SOURCE, {s: sum(len(v) for v in p.values()) for s, p in pools.items()}, 'pairs')


In [ ]:
# Load the judge (fp16, left-padding for batched next-token logits).
tok = AutoTokenizer.from_pretrained(cfg.llm_ckpt, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(cfg.llm_ckpt, torch_dtype=torch.float16,
                                           device_map='auto').eval()
print('judge loaded')


In [ ]:
# Score every (topic, doc) on R; write the feature jsonl. Resumable-friendly: one line per pair.
from tqdm.auto import tqdm
with open(OUT_PATH, 'w') as out:
    for s in SETS:
        for t, docs in tqdm(pools[s].items(), desc=f'judge {s}'):
            scores = llm_yesno_scores(llm, tok, sets[s]['topic2text'][t],
                                      [id2fields[d] for d in docs], cfg, batch=8)
            for d, sc in zip(docs, scores):
                out.write(json.dumps({'source': s, 'topic_id': t, 'doc_id': d,
                                      'llm_score': float(sc)}) + '\n')
print('wrote', OUT_PATH)


In [ ]:
# Standalone judged-pool NDCG@10 under R (does reading eligibility help the judge?).
scores = {}
for line in open(OUT_PATH):
    r = json.loads(line); scores.setdefault((r['source'], r['topic_id']), {})[r['doc_id']] = r['llm_score']
rows = []
for s in SETS:
    rel = sets[s]['rel_dict']; vals = []
    for t, docs in pools[s].items():
        ranked = sorted(docs, key=lambda d: scores[(s, t)][d], reverse=True)
        vals.append(ndcg_at_k(ranked, rel[t]))
    rows.append({'split': s, f'judge_ndcg@10 ({POOL_TAG})': round(float(np.mean(vals)), 4)})
pd.DataFrame(rows)


## Reading it
- Compare the standalone judge NDCG under R against §7f (judge on eligibility-only text, saw eligibility)
  and against the eligibility-blind ensemble feature §11c was built on. If R clearly helps, §11c's null was
  partly an **input artifact**, and the judge is worth re-testing in the ensemble (and possibly re-fine-tuning).
- `llm_scores_R.jsonl` is the `llm_yesno` feature for `train_ensemble_full` on R.
